# Comparaison de transcription Darija — MoulSot vs Whisper

Une vidéo courte (lien Drive **public**) → le même découpage en phrases pour tous les modèles → un tableau phrase par phrase.

| Colonne | Modèle |
|---|---|
| `MOULSOT` | MoulSot v0.3 (Qwen3-ASR 1.7B, Darija + français/anglais), langue « Arabic » |
| `MOULSOT_AUTO` | MoulSot v0.3, langue détectée automatiquement |
| `TURBO_AR` | Whisper large-v3-turbo de base, arabe |
| `DARIJA` | Whisper turbo + Darija (`anaszil/whisper-large-v3-turbo-darija`) = **notre méthode actuelle** |
| `TURBO_EN` / `TURBO_FR` | Whisper turbo forcé anglais / français = **notre passe EN/FR actuelle** |

**Avant de lancer :** Runtime → Change runtime type → **T4 GPU**, puis lance les cellules 1 → 5 dans l'ordre.
Aucun redémarrage n'est nécessaire : chaque modèle tourne dans son propre processus.
À la fin, `comparaison.txt` se télécharge : envoie-le à Claude.

In [ ]:
#@title 1 · Réglages
#@markdown Lien Google Drive de la vidéo (partagée « Tous les utilisateurs disposant du lien »).
VIDEO_LINK = ""  #@param {type:"string"}
#@markdown Ne garder que les N premières minutes (0 = toute la vidéo).
MAX_MINUTES = 3  #@param {type:"number"}
#@markdown Longueur maximale d'une phrase (secondes), coupée aux pauses.
PHRASE_MAX_S = 15  #@param {type:"number"}
#@markdown Optionnel : mots du sujet pour aider MoulSot (ex. « cybersécurité, Python, Progma »).
CONTEXT = ""  #@param {type:"string"}
W = "/content/cmp"

In [ ]:
#@title 2 · Télécharger la vidéo + découper en phrases
!pip -q install gdown faster-whisper
!pip -q uninstall -y torchao
import os, json, subprocess, numpy as np, gdown
from faster_whisper.vad import get_speech_timestamps, VadOptions
assert VIDEO_LINK.strip(), "Mets le lien de la vidéo dans la cellule 1."
os.makedirs(W, exist_ok=True)
print("GPU :", subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip() or "AUCUN — active le T4 !")

src = gdown.download(VIDEO_LINK.strip(), f"{W}/video", fuzzy=True, quiet=False)
assert src and os.path.getsize(src) > 0, "Téléchargement raté : la vidéo est-elle partagée « Tous les utilisateurs disposant du lien » ?"
cmd = ["ffmpeg", "-v", "error", "-i", src, "-vn", "-ac", "1", "-ar", "16000", "-f", "f32le", "-"]
if MAX_MINUTES and MAX_MINUTES > 0:
    cmd[3:3] = ["-t", str(MAX_MINUTES * 60)]
audio = np.frombuffer(subprocess.run(cmd, check=True, capture_output=True).stdout, np.float32)
np.save(f"{W}/audio.npy", audio)

SR = 16000
ch = get_speech_timestamps(audio, VadOptions(min_silence_duration_ms=300, speech_pad_ms=120, max_speech_duration_s=PHRASE_MAX_S))
spans = []
for c in ch:
    s, e = c["start"] / SR, c["end"] / SR
    if spans and s - spans[-1][1] < 1.0 and e - spans[-1][0] <= PHRASE_MAX_S:
        spans[-1] = (spans[-1][0], e)
    else:
        spans.append((s, e))
json.dump({"phrases": [[round(s, 2), round(e, 2)] for s, e in spans], "context": CONTEXT}, open(f"{W}/phrases.json", "w"))
print(f"\nAudio : {len(audio) / SR / 60:.1f} min · {len(spans)} phrases")

In [ ]:
#@title 3 · Whisper : turbo (AR / EN / FR) + turbo Darija (notre méthode actuelle)
script = r"""
import json, time, numpy as np, torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
W = "/content/cmp"; SR = 16000
A = np.load(f"{W}/audio.npy"); P = json.load(open(f"{W}/phrases.json"))["phrases"]
dev = "cuda"; dt = torch.float16
BASE = "openai/whisper-large-v3-turbo"; LORA = "anaszil/whisper-large-v3-turbo-darija"
proc = WhisperProcessor.from_pretrained(BASE)
model = WhisperForConditionalGeneration.from_pretrained(BASE, torch_dtype=dt).to(dev).eval()

def run(name, lang):
    t0 = time.time(); out = []
    for s, e in P:
        x = A[int(s * SR):int(e * SR)]
        enc = proc(x, sampling_rate=SR, return_tensors="pt", return_attention_mask=True)
        with torch.no_grad():
            ids = model.generate(enc.input_features.to(dev, dt), attention_mask=enc.attention_mask.to(dev),
                                 language=lang, task="transcribe", num_beams=1)
        out.append(proc.batch_decode(ids, skip_special_tokens=True)[0].strip())
    took = time.time() - t0
    print(f"{name}: {took:.0f} s", flush=True)
    return {"lines": out, "seconds": round(took, 1)}

res = {}
res["TURBO_AR"] = run("TURBO_AR", "ar")
res["TURBO_EN"] = run("TURBO_EN", "en")
res["TURBO_FR"] = run("TURBO_FR", "fr")
try:
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, LORA).merge_and_unload().to(dev, dt).eval()
    res["DARIJA"] = run("DARIJA", "ar")
except Exception as e:
    print("!! Darija LoRA non chargé :", str(e)[:300])
json.dump(res, open(f"{W}/whisper.json", "w"), ensure_ascii=False)
print("OK whisper.json")
"""
open(f"{W}/run_whisper.py", "w").write(script)
!pip -q install peft accelerate
!python {W}/run_whisper.py

In [ ]:
#@title 4 · MoulSot v0.3
script = r"""
import json, time, numpy as np, torch
from qwen_asr import Qwen3ASRModel
W = "/content/cmp"; SR = 16000
A = np.load(f"{W}/audio.npy"); J = json.load(open(f"{W}/phrases.json")); P = J["phrases"]
dt = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
model = None
for mid in ("atlasia/moulsot.v0.3", "01Yassine/moulsot.v0.3"):
    try:
        model = Qwen3ASRModel.from_pretrained(mid, dtype=dt, device_map="cuda", max_inference_batch_size=8, max_new_tokens=300); break
    except Exception as e:
        print("!!", mid, str(e)[:200])
items = [(A[int(s * SR):int(e * SR)], SR) for s, e in P]
res = {}
for name, lang in (("MOULSOT", "Arabic"), ("MOULSOT_AUTO", None)):
    try:
        t0 = time.time()
        r = model.transcribe(audio=items, context=J.get("context", ""), language=lang)
        res[name] = {"lines": [(x.text or "").strip() for x in r], "seconds": round(time.time() - t0, 1)}
        print(f"{name}: {res[name]['seconds']:.0f} s", flush=True)
    except Exception as e:
        print(f"!! {name} :", str(e)[:300])
json.dump(res, open(f"{W}/moulsot.json", "w"), ensure_ascii=False)
print("OK moulsot.json")
"""
open(f"{W}/run_moulsot.py", "w").write(script)
!pip -q install qwen-asr
!python {W}/run_moulsot.py

In [ ]:
#@title 5 · Tableau de comparaison + téléchargement
import json, os
from google.colab import files
P = json.load(open(f"{W}/phrases.json"))["phrases"]
R = {}
for f in ("moulsot.json", "whisper.json"):
    if os.path.exists(f"{W}/{f}"): R.update(json.load(open(f"{W}/{f}")))
ORDER = [k for k in ("MOULSOT", "MOULSOT_AUTO", "DARIJA", "TURBO_AR", "TURBO_EN", "TURBO_FR") if k in R]
L = ["Temps de calcul (T4) : " + " · ".join(f"{k} {R[k]['seconds']:.0f} s" for k in ORDER), ""]
for i, (s, e) in enumerate(P):
    L.append(f"#{i + 1:03d}  {s:7.2f} - {e:7.2f}")
    for k in ORDER:
        L.append(f"   {k:<13}| {R[k]['lines'][i]}")
    L.append("")
txt = "\n".join(L)
print(txt)
open(f"{W}/comparaison.txt", "w", encoding="utf-8").write(txt)
files.download(f"{W}/comparaison.txt")